# SARIMAX nacional — 16 tipologias simultâneas

Esta análise foi executada no script `src/sarimax_revisao_final.py`. O notebook examina e interpreta as saídas salvas. Para reestimar os modelos, execute o script conforme a seção de reprodução. As seleções não utilizam a amostra de teste.

## 1. Protocolo de estimação

Resposta: `100 Δlog(I)`. Exógenas: `Δlog(1+D)`. SARIMA e SARIMAX usam a mesma ordem e sazonalidade anual `s=12` no par; a escolha é feita no treino. Avaliação condicional de um passo com `x_t` conhecida.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
root=Path.cwd()
if not (root/'outputs'/'tables').exists(): root=root.parent
tables=root/'outputs'/'tables'/'sarimax_revisao_20261019'


In [2]:
df=pd.read_csv(tables/'nacional_comparacao_modelos.csv')
audit=pd.read_csv(tables/'nacional_auditoria_posto.csv')
coef=pd.read_csv(tables/'nacional_coeficientes.csv')
summary=json.loads((tables/'nacional_resumo.json').read_text(encoding='utf-8'))
print('Resumo:', summary)
print('Postos originais:', audit.groupby('periodo')[['posto_treino','posto_completo']].agg(['min','max']).to_string())

Resumo: {'amostra': '2013-2019/2020 e 2013-2024', 'modo': 'nacional', 'n_modelos': 2, 'n_comparaveis': 2, 'n_melhor_rmse': 0, 'n_melhor_bic': 0, 'n_LB_sarimax_nao_rejeita': 0, 'n_LB_sarima_nao_rejeita': 1, 'n_rejeicoes_conjunto_BH': 0, 'n_rejeicoes_coef_global_BH': 0, 'mediana_delta_RMSE_pct': 19.79604788857602, 'criterio_selecao': 'apenas treino: suporte>=3 meses, posto completo, cond<=1000 e cap 9 ou 12 (estaduais)', 'exog_t': 'Δlog(1+D), contemporâneo, conhecido no teste', 'y': '100 Δlog(CI/CA), Brasil calculado por soma das carteiras'}
Postos originais:         posto_treino     posto_completo    
                 min max            min max
periodo                                    
Pré               16  16             16  16
Total             16  16             16  16


## 2. Resultados e comparação

Δ RMSE negativo indica melhora do SARIMAX. BIC comparado apenas para modelos ajustados no mesmo treino.

In [3]:
cols=['periodo','unidade','n_exog_incluidas','sazonal','rmse_sarima','rmse_sarimax','variacao_rmse_pct','delta_bic_treino','p_conjunto_treino','q_conjunto_bh','lb_sarimax_treino']
print(df[cols].to_string(index=False))
print('Modelos comparáveis:',df.rmse_sarimax.notna().sum(),'Melhora RMSE:',(df.variacao_rmse_pct<0).sum(),'Melhora BIC:',(df.delta_bic_treino<0).sum())

periodo unidade  n_exog_incluidas       sazonal  rmse_sarima  rmse_sarimax  variacao_rmse_pct  delta_bic_treino  p_conjunto_treino  q_conjunto_bh  lb_sarimax_treino
    Pré  Brasil                16 (1, 0, 0, 12)     1.668625      2.152674          29.008919         57.207935           0.795788       0.795788           0.004821
  Total  Brasil                16 (1, 0, 0, 12)     1.354183      1.497498          10.583177         64.107716           0.718972       0.795788           0.026136
Modelos comparáveis: 2 Melhora RMSE: 0 Melhora BIC: 0


## 3. Evidências e limites

Os p-valores são condicionais à seleção do modelo e não representam causalidade. Os regressors `x_t` contemporâneos são conhecidos no teste; a previsão é condicional. Verificar o Ljung–Box e a família de multiplicidade ao interpretar efeitos individuais.

In [4]:
if 'q_bh_global' in coef.columns:
    print(coef.loc[coef.q_bh_global<.05,['periodo','unidade','tipologia','coef_original','p','q_bh_global']].to_string(index=False))
else: print('Sem coeficientes elegíveis ao BH global')

Empty DataFrame
Columns: [periodo, unidade, tipologia, coef_original, p, q_bh_global]
Index: []


## 4. Reprodução integral

No terminal do projeto: `python src/sarimax_revisao_final.py nacional`. Consulte `docs/sarimax_revisao_estadual_nacional_20261019.md` para definições, seleção de suporte, limitações e conclusões.